# DHL Pickup Optimization: Customer Segmentation using RFM + K-Means Clustering
**Topic: Unsupervised Learning – Clustering (K-Means)**


In [ ]:
# --- Import Libraries ---
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
# --- Load the Dataset ---
# This dataset simulates DHL customer pickup transactions
df = pd.read_excel("Online Retail.xlsx")  # Replace with your dataset path if needed

In [ ]:
# --- Preprocessing ---
# Remove transactions with missing customer IDs
df = df.dropna(subset=['CustomerID'])

# Convert InvoiceDate to proper datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Optional: Add TotalPrice column for clarity
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Set a snapshot date for Recency calculation
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

In [ ]:
# --- RFM Feature Engineering ---
# RFM = Recency, Frequency, Monetary
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'TotalPrice': 'sum'                                       # Monetary
}).reset_index()

# Rename columns for clarity
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

In [ ]:
# --- Standardize Features ---
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

In [ ]:
# --- Apply K-Means Clustering ---
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
# --- Visualize the Clusters ---
plt.figure(figsize=(8, 6))
plt.scatter(rfm['Recency'], rfm['Monetary'], c=rfm['Cluster'], cmap='viridis')
plt.title("DHL Customer Segments based on RFM Clustering")
plt.xlabel("Recency (days since last pickup)")
plt.ylabel("Monetary Value (Total Spend)")
plt.grid(True)
plt.colorbar(label='Cluster ID')
plt.show()

### Optional Enhancements for Learners
- Try the Elbow Method to find the optimal number of clusters
- Use Silhouette Score to validate clustering performance
- Perform dimensionality reduction (e.g., PCA) for better visualization

### Real-World Use Case
**DHL can use this segmentation to:**
- Prioritize high-value frequent customers for express pickups
- Identify churn-risk customers (high recency, low frequency)
- Tailor promotions or loyalty programs to specific segments